In [36]:
import json, re, random, os
import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import Path
from typing import List, Dict, Any
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

CHUNKS_PATH = "chunks.paragraphs.jsonl"

In [40]:
'''Chunking Display'''

# Peek raw
print("abs path:", os.path.abspath(CHUNKS_PATH))
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    raw = [ln for ln in f if ln.strip()][:3]
print("first 3 lines (raw):")
for ln in raw:
    print(ln[:200].replace("\n"," ") + ("..." if len(ln) > 200 else ""))

# Load rows
rows = [json.loads(ln) for ln in open(CHUNKS_PATH, "r", encoding="utf-8") if ln.strip()]
print("rows:", len(rows))
print("keys example:", sorted(rows[0].keys()))

# Score string-like fields
lens = defaultdict(list)
for r in rows:
    for k,v in r.items():
        if isinstance(v, str):
            lens[k].append(len(v.strip()))

def score(arr): 
    return (sum(1 for x in arr if x>0), round(statistics.mean(arr),1) if arr else 0)

candidates = {k: score(arr) for k,arr in lens.items()}
print("string fields (nonempty_count, avg_len):")
for k,(n,avg) in sorted(candidates.items(), key=lambda x: (x[1][0], x[1][1]), reverse=True):
    print(f"- {k}: {n}, {avg}")

# Choose the best text key
TEXT_KEY = max(candidates, key=lambda k: (candidates[k][0], candidates[k][1])) if candidates else None
print("Chosen TEXT_KEY:", TEXT_KEY)

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

N = 5
examples = []
for i, r in enumerate(rows[:N]):
    txt = r.get(TEXT_KEY, "")
    emb = model.encode([txt], normalize_embeddings=True)[0]
    examples.append({
        "chunk_id": r.get("id", f"chunk-{i}"),
        "content_preview": (txt or "")[:200].replace("\n"," "),
        "embedding_dim": len(emb),
        "embedding_vector_first5": emb[:5].round(4).tolist(),
    })

pd.DataFrame(examples)

abs path: /Users/matteogevi/Aurora-History-MVP/notebooks/chunks.paragraphs.jsonl
first 3 lines (raw):
{"chunk_id": "b203c98fadb3f0fe::root::c000001", "chunk_seq": 1, "section_id": null, "level": null, "text": "“This book of fers a comprehensive, well-structured guide to the essential  \naspects of bui...
{"chunk_id": "b203c98fadb3f0fe::root::c000002", "chunk_seq": 2, "section_id": null, "level": null, "text": "_**Building Applications with Foundation Models**_  \n_**Chip Huyen**_  \n**AI Engineering**...
{"chunk_id": "b203c98fadb3f0fe::root::c000003", "chunk_seq": 3, "section_id": null, "level": null, "text": "Answer:", "doc_key": "b203c98fadb3f0fe", "level_path": "", "version": 1} 
rows: 260
keys example: ['chunk_id', 'chunk_seq', 'doc_key', 'level', 'level_path', 'section_id', 'text', 'version']
string fields (nonempty_count, avg_len):
- text: 260, 4402.3
- chunk_id: 260, 31
- doc_key: 260, 16
- level_path: 0, 0
Chosen TEXT_KEY: text


,chunk_id,content_preview,embedding_dim,embedding_vector_first5
0,chunk-0,"“This book of fers a comprehensive, well-struc...",384,"[-0.07530000060796738, -0.051899999380111694, ..."
1,chunk-1,_**Building Applications with Foundation Model...,384,"[-0.03290000185370445, -0.05609999969601631, -..."
2,chunk-2,Answer:,384,"[-0.029200000688433647, 0.11110000312328339, -..."
3,chunk-3,``` A language model might be able to comple...,384,"[-0.054099999368190765, -0.10599999874830246, ..."
4,chunk-4,"``` The query, key, and value matrices have ...",384,"[0.01850000023841858, -0.0414000004529953, -0...."


In [44]:
'''Chunk length & duplicate sniff'''

rows = [json.loads(l) for l in open(CHUNKS_PATH, "r", encoding="utf-8") if l.strip()]
print("rows loaded:", len(rows))

# --- auto-detect TEXT_KEY (string field with most non-empty content and largest avg len) ---
lens = defaultdict(list)
for r in rows:
    for k, v in r.items():
        if isinstance(v, str):
            lens[k].append(len(v.strip()))

def score(arr):
    nonempty = sum(1 for x in arr if x > 0)
    avg = statistics.mean(arr) if arr else 0
    return (nonempty, avg)

candidates = {k: score(arr) for k, arr in lens.items()}
TEXT_KEY = max(candidates, key=lambda k: (candidates[k][0], candidates[k][1]))
print("Using TEXT_KEY:", TEXT_KEY, "->", candidates[TEXT_KEY])

# --- chunk length & duplicate sniff using TEXT_KEY ---
lengths = [len((r.get(TEXT_KEY) or "").strip()) for r in rows]
print(
    "n:", len(rows),
    "| avg chars:", round(statistics.mean(lengths), 1) if lengths else 0,
    "| median:", int(statistics.median(lengths)) if lengths else 0,
    "| >2000 chars:", sum(int(L > 2000) for L in lengths)
)

def normalize(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").strip().lower())

seen = set()
dups = 0
for r in rows:
    key = normalize(r.get(TEXT_KEY) or "")[:400]  # prefix to keep memory low
    if key in seen:
        dups += 1
    seen.add(key)

print("potential duplicates:", dups)


rows loaded: 260
Using TEXT_KEY: text -> (260, 4402.315384615385)
n: 260 | avg chars: 4402.3 | median: 62 | >2000 chars: 35
potential duplicates: 15


In [23]:
'''Embedding norms + self-nearest-neighbor sanity'''

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

texts = [r.get("content","") for r in rows[:500]]  # subset for speed
X = model.encode(texts, normalize_embeddings=True).astype("float32")

# Check unit norms
norms = np.linalg.norm(X, axis=1)
print("norm mean:", float(norms.mean()), "min:", float(norms.min()), "max:", float(norms.max()))

# Self NN (exclude self)
S = X @ X.T
np.fill_diagonal(S, -1.0)
top_idx = S.argmax(axis=1)
sample = [(i, int(top_idx[i]), float(S[i, top_idx[i]])) for i in range(min(10, len(texts)))]
print("top neighbors (i -> j, cos):", sample[:5])

norm mean: 0.9999998807907104 min: 0.9999999403953552 max: 1.0
top neighbors (i -> j, cos): [(0, 93, 1.0000001192092896), (1, 93, 1.0000001192092896), (2, 93, 1.0000001192092896), (3, 93, 1.0000001192092896), (4, 93, 1.0000001192092896)]


In [27]:
'''“Same section” vs random similarity'''

sec = [r.get("section_node_id") for r in rows[:500]]
pairs_same, sims_same = 0, []
pairs_diff, sims_diff = 0, []

for _ in range(100):
    i = random.randrange(len(sec)); j = random.randrange(len(sec))
    if i == j: continue
    sim = float(X[i] @ X[j])
    if sec[i] and sec[i] == sec[j]:
        pairs_same += 1; sims_same.append(sim)
    else:
        pairs_diff += 1; sims_diff.append(sim)

def m(a): return round(sum(a)/len(a), 4) if a else None
print("same-sec avg cos:", m(sims_same), "n:", pairs_same)
print("diff-sec avg cos:", m(sims_diff), "n:", pairs_diff)

same-sec avg cos: None n: 0
diff-sec avg cos: 1.0 n: 97
